Judge saved model responses and record case-level scores, criterion decisions and token usage.


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import hashlib
import json
import re
import time
from typing import Any, Dict, List
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm


In [ ]:
MODEL = 'gpt-4.1-mini-2025-04-14'
API_KEY = ''
TEMPERATURE_JUDGE = 0.3
MAX_TOKENS_JUDGE = 500
REQUEST_TIMEOUT = 60
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
INPUT_CSV = 'delphi_pubmedqa_unlabeled_noroles.csv'
OUTPUT_CSV = 'delphi_pubmedqa_unlabeled_judge.csv'
ANSWER_COLUMN = 'winner_answer_text'
REQUIRED_COLUMNS = ['question', 'context', 'long_answer', ANSWER_COLUMN]

def load_dataset(path: str) -> pd.DataFrame:
    d = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    d.columns = [c.lstrip('\ufeff') for c in d.columns]
    missing = [c for c in REQUIRED_COLUMNS if c not in d.columns]
    if missing:
        raise ValueError(f'Required column(s) missing in {path}: {missing}')
    d = d[d[ANSWER_COLUMN].notna() & (d[ANSWER_COLUMN].astype(str).str.strip() != '')]
    d = d.reset_index(drop=True)
    return d
print(f'Judging: {INPUT_CSV}  ->  {OUTPUT_CSV}')
print(f'Answer column: {ANSWER_COLUMN}')


In [ ]:
judge_client = OpenAI(api_key=API_KEY)
judge_cache: Dict[str, Dict[str, Any]] = {}
_RESPONSE_FORMAT = {'type': 'json_schema', 'json_schema': {'name': 'answer_grading', 'strict': True, 'schema': {'type': 'object', 'properties': {'explanation': {'type': 'string'}, 'score': {'type': 'integer'}}, 'required': ['explanation', 'score'], 'additionalProperties': False}}}

def _make_cache_key(system_text: str, user_text: str) -> str:
    raw = json.dumps({'model': MODEL, 'system': system_text, 'user': user_text}, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def call_judge(system_text: str, user_text: str) -> Dict[str, Any]:
    cache_key = _make_cache_key(system_text, user_text)
    if cache_key in judge_cache:
        return {'text': judge_cache[cache_key]['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    last_error = ''
    for attempt in range(MAX_RETRIES):
        try:
            response = judge_client.chat.completions.create(model=MODEL, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=TEMPERATURE_JUDGE, max_tokens=MAX_TOKENS_JUDGE, response_format=_RESPONSE_FORMAT, timeout=REQUEST_TIMEOUT)
            text = response.choices[0].message.content or ''
            usage = response.usage
            judge_cache[cache_key] = {'text': text}
            return {'text': text, 'prompt_tokens': int(getattr(usage, 'prompt_tokens', 0) or 0), 'completion_tokens': int(getattr(usage, 'completion_tokens', 0) or 0), 'total_tokens': int(getattr(usage, 'total_tokens', 0) or 0), 'cached': False}
        except Exception as e:
            last_error = str(e)
            wait = BASE_SLEEP_SECONDS * 2 ** attempt
            m = re.search('try again in ([0-9.]+)s', last_error)
            time.sleep(float(m.group(1)) + 0.5 if m else wait)
    raise RuntimeError(f'Judge call failed after {MAX_RETRIES} retries. Last error: {last_error}')


In [ ]:
_JUDGE_SYSTEM_PROMPT = 'You are an expert biomedical answer grader.\nYour task is to assess how well a candidate answer to a research question matches the correct\nreference conclusion, given the supporting context.\n\nScore the candidate answer\'s correctness and semantic similarity to the reference conclusion on an\ninteger scale from 1 to 10:\n  1    = completely wrong, or contradicts the reference conclusion\n  5-6  = partially correct: captures some but not all of the reference conclusion, or is vague/hedged\n         where the reference is specific\n  10   = fully correct: equivalent in meaning to the reference conclusion, even if the wording differs\n\nJudge substance, not style: differences in phrasing, length, or formatting must not affect the score.\nBase your judgment on the reference conclusion and the given context. Do not penalise the candidate\nfor omitting details that are themselves absent from the reference conclusion.\nReturn JSON only with fields "explanation" (one concise sentence) and "score" (integer 1-10).'.strip()

def build_grading_user_prompt(question: str, context: str, reference_answer: str, candidate_answer: str) -> str:
    return f'Research question:\n<<<QUESTION>>>\n{question}\n<<<END QUESTION>>>\n\nContext:\n<<<CONTEXT>>>\n{context}\n<<<END CONTEXT>>>\n\nReference conclusion (correct answer):\n<<<REFERENCE>>>\n{reference_answer}\n<<<END REFERENCE>>>\n\nCandidate answer to grade:\n<<<CANDIDATE>>>\n{candidate_answer}\n<<<END CANDIDATE>>>\n'.strip()

def _int_in_range(value: Any, lo: int, hi: int, default: int) -> int:
    try:
        v = int(value)
    except Exception:
        return default
    return max(lo, min(hi, v))

def parse_judge_output(text: str) -> Dict[str, Any]:
    raw = str(text).strip() if text else ''
    try:
        data = json.loads(raw)
        return {'score': _int_in_range(data.get('score'), 1, 10, 1), 'explanation': str(data.get('explanation', '')).strip()}
    except Exception:
        pass
    m_score = re.search('"score"\\s*:\\s*(-?\\d+)', raw)
    if m_score:
        return {'score': _int_in_range(m_score.group(1), 1, 10, 1), 'explanation': raw}
    raise ValueError(f'Could not parse judge output: {raw[:200]}')


In [ ]:
def run_judge_for_case(question: str, context: str, reference_answer: str, candidate_answer: str) -> Dict[str, Any]:
    user_text = build_grading_user_prompt(question, context, reference_answer, candidate_answer)
    response = call_judge(_JUDGE_SYSTEM_PROMPT, user_text)
    parsed = parse_judge_output(response['text'])
    return {'Judge_score_1_10': parsed['score'], 'Judge_explanation': parsed['explanation'], 'judge_prompt_tokens': response['prompt_tokens'], 'judge_completion_tokens': response['completion_tokens'], 'judge_total_tokens': response['total_tokens'], 'judge_cached': response['cached']}


In [ ]:
def run_judge_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for _, row in tqdm(input_df.iterrows(), total=len(input_df), desc='Judge'):
        row_data = row.to_dict()
        question = str(row_data.get('question', '') or '')
        context = str(row_data.get('context', '') or '')
        reference = str(row_data.get('long_answer', '') or '')
        candidate = str(row_data.get(ANSWER_COLUMN, '') or '')
        if not reference.strip() or not candidate.strip():
            row_data['Judge_score_1_10'] = None
            row_data['Judge_explanation'] = ''
            row_data['judge_prompt_tokens'] = 0
            row_data['judge_completion_tokens'] = 0
            row_data['judge_total_tokens'] = 0
            row_data['judge_cached'] = False
            rows.append(row_data)
            continue
        result = run_judge_for_case(question, context, reference, candidate)
        row_data.update(result)
        rows.append(row_data)
    return pd.DataFrame(rows)
df = load_dataset(INPUT_CSV)
print('Rows to judge:', len(df))
result_df = run_judge_pipeline(df)
result_df.to_csv(OUTPUT_CSV, sep=';', index=False, encoding='utf-8-sig')
print()
scores = result_df['Judge_score_1_10'].dropna()
if len(scores):
    print(f'Judge_score_1_10  mean={scores.mean():.2f}  std={scores.std():.2f}  min={scores.min():.0f}  max={scores.max():.0f}  n={len(scores)}')
tt = int(result_df['judge_total_tokens'].sum())
print(f'total_tokens: {tt:,}')
print(f'saved: {OUTPUT_CSV}  ({len(result_df)} rows)')
